# Employees: Raw -> Bronze

Land the raw Excel extract as-is, only standardizing column names.

In [1]:
%run ../00_config.ipynb
%run ../00_utils.ipynb

/usr/local/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


[08/23/26 15:05:49] INFO     Using                                                                  ]8;id=13662410;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=13662411;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py#302\302]8;;\
                             '/usr/local/lib/python3.12/site-packages/kedro/framework/project/rich_                
                             logging.yml' as logging configuration.                                                

[08/23/26 15:05:49] WARNING  /usr/local/lib/python3.12/site-packages/kedro/framework/context/contex ]8;id=13662418;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=13662419;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             t.py:221: UserWarning: Parameters not found in your Kedro project                     
                             config.                                                                               
                             No files of YAML or JSON format found in /app/conf/base or                            
                             /app/conf/local matching the glob pattern(s): ['parameters*',                         
                             'parameters*/**', '**/parameters*']                                                   
                               warn(f"Parameters not found in your Kedro project config.\n{exc!s}")                
                                                                                                                   

                    INFO     No typed parameter requirements found, returning original   ]8;id=13662426;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=13662427;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py#124\124]8;;\
                             parameters                                                                            

Kedro context loaded from /app
Catalog datasets: ['raw_employees', 'bronze_employees', 'silver_employees', 'gold_employees', 'raw_sales', 'bronze_sales', 'silver_sales', 'gold_sales', 'raw_inventory', 'bronze_inventory', 'silver_inventory', 'gold_inventory', 'parameters']


[08/23/26 15:05:50] WARNING  /usr/local/lib/python3.12/site-packages/nbformat/validator.py:434:     ]8;id=13662432;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=13662433;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             MissingIDFieldWarning: Cell is missing an id field, this will become a                
                             hard error in future nbformat versions. You may want to use                           
                             `normalize()` on your notebooks before validations (available since                   
                             nbformat 5.1.4). Previous versions of nbformat are fixing this issue                  
                             transparently, and will stop doing so in the future.                                  
                               _validate(nbdict, ref, version, version_minor, relax_add_props)                     
                                                                                                                   

In [2]:
raw_employees = catalog.load("raw_employees")
raw_employees

                    INFO     Loading data from raw_employees (ExcelDataset)...                 ]8;id=13662440;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=13662441;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1050\1050]8;;\

,Employee ID,Full Name,Department,Salary,Hire Date,Internal Notes
0,101.0,Alice Johnson,engineering,95000,2021-03-15,top performer
1,102.0,Bob Smith,SALES,72000,2020-07-01,NaN
2,103.0,Carla Diaz,engineering,88000,2022-01-10,remote
3,NaN,Ghost Row,unknown,0,2019-05-05,"bad data, should be dropped"
4,104.0,David Lee,marketing,68000,2023-11-20,NaN


In [3]:
bronze_employees = standardize_columns(raw_employees)
bronze_employees

,employee_id,full_name,department,salary,hire_date,internal_notes
0,101.0,Alice Johnson,engineering,95000,2021-03-15,top performer
1,102.0,Bob Smith,SALES,72000,2020-07-01,NaN
2,103.0,Carla Diaz,engineering,88000,2022-01-10,remote
3,NaN,Ghost Row,unknown,0,2019-05-05,"bad data, should be dropped"
4,104.0,David Lee,marketing,68000,2023-11-20,NaN


In [4]:
catalog.save("bronze_employees", bronze_employees)

                    INFO     Saving data to bronze_employees (CSVDataset)...                   ]8;id=13662447;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=13662448;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1006\1006]8;;\

## PySpark alternative (reference only)

PySpark isn't installed in this image. Left commented out to show how this
stage would read/write with Spark instead of pandas.

In [5]:
# raw_employees_spark = (
#     spark.read.format("com.crealytics.spark.excel")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load(str(PROJECT_ROOT / "data/01_raw/employees.xlsx"))
# )
#
# bronze_employees_spark = standardize_columns_spark(raw_employees_spark)
#
# bronze_employees_spark.write.mode("overwrite").option("header", "true").csv(
#     str(PROJECT_ROOT / "data/02_bronze/employees.csv")
# )